# Audit di coerenza NFT — JSONL normalizzati

Adattamento del notebook `pulizia_history.ipynb` al formato **una action per riga**. Non esiste più la colonna `history`: si confronta `nft` con i riferimenti espliciti nei `details`.

**Solo audit:** non elimina, modifica o sposta alcun evento. I campi `old_owner`, `new_owner`, `payout_address`, `marketplace_address` sono wallet/contratti e **non** sono NFT estranei.

Controlla i due layer insieme, classifica il target dell'action se differente dal NFT principale e produce CSV di report.

In [ ]:
import json, csv
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from pathlib import Path

BASE = Path('..')

FILES = {
    'usernames': BASE / 'dataset_normalized/usernames_normalized.jsonl',
    'phone_numbers': BASE / 'dataset_normalized/numbers_normalized.jsonl',
}

OUT = BASE / 'cleaning/history_audit'
OUT.mkdir(parents=True, exist_ok=True)

def addr(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    return x or None

# Nei details delle azioni NFT di TON Center, questi campi rappresentano NFT.
# NON includere owner/sender/recipient, che sono indirizzi di wallet o contratti.
NFT_FIELDS = ('nft_item', 'nft_address')

for layer, path in FILES.items():
    print(layer, '->', path.resolve(), '| esiste:', path.is_file())

usernames -> /home/local/ADUNIPI/a.belliani/CNA/dataset_normalized/usernames_normalized.jsonl | esiste: True
phone_numbers -> /home/local/ADUNIPI/a.belliani/CNA/dataset_normalized/numbers_normalized.jsonl | esiste: True


## 1. Indicizzare gli asset dei due dataset

Scansione sequenziale; conserva in RAM **solo gli indirizzi NFT distinti**, non i record. Serve a distinguere i target estranei presenti nell'altro layer da quelli non presenti nei file filtrati.

In [2]:
asset_layers = defaultdict(set)
counts = Counter()
for layer, path in FILES.items():
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                counts[(layer, 'invalid_json')] += 1
                continue
            nft = addr(record.get('nft', record.get('n')))
            if nft:
                asset_layers[nft].add(layer)
            else:
                counts[(layer, 'missing_row_nft')] += 1
            counts[(layer, 'rows')] += 1

print('NFT distinti (unione dei layer):', len(asset_layers))
print('NFT presenti in entrambi i layer:', sum(len(x)>1 for x in asset_layers.values()))
for layer in FILES:
    print(layer, 'righe:', counts[(layer,'rows')], '| JSON non validi:', counts[(layer,'invalid_json')])

NFT distinti (unione dei layer): 691773
NFT presenti in entrambi i layer: 0
usernames righe: 2579153 | JSON non validi: 0
phone_numbers righe: 837146 | JSON non validi: 0


## 2. Audit di `details` per ogni action

- **MATCH:** riferimento NFT coerente con `nft` della riga;
- **MISMATCH:** `details` contiene un riferimento esplicito a un altro NFT;
- **NO_NFT_REFERENCE:** nessuno dei campi NFT previsti è presente; **non è un'anomalia provata**;
- **CONFLICTING_NFT_FIELDS:** due campi NFT espliciti discordano nello stesso `details`.

Registra soltanto righe sospette in CSV, con gli identificativi per rintracciarle, senza duplicare l'intero `details`. Non cambia i dati.

In [3]:
report_path = OUT / 'nft_mismatch_report.csv'
fieldnames = ['layer','line_number','row_nft','asset_name','action_type','action_id','trace_id',
              'timestamp','details_nft_item','details_nft_address','status',
              'target_in_dataset','target_layers','target_nft']

stats = Counter()
examples = defaultdict(list)
with report_path.open('w', encoding='utf-8', newline='') as report:
    writer = csv.DictWriter(report, fieldnames=fieldnames)
    writer.writeheader()
    for layer, path in FILES.items():
        with path.open('r', encoding='utf-8') as fin:
            for line_no, line in enumerate(fin, 1):
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    stats[(layer,'INVALID_JSON')] += 1
                    continue
                row_nft = addr(r.get('nft', r.get('n')))
                d = r.get('details', r.get('det')) or {}
                if not isinstance(d, dict):
                    stats[(layer,'INVALID_DETAILS')] += 1
                    continue
                refs = {k:addr(d.get(k)) for k in NFT_FIELDS if addr(d.get(k))}
                distinct_refs = set(refs.values())
                if len(distinct_refs) > 1:
                    status = 'CONFLICTING_NFT_FIELDS'
                elif not distinct_refs:
                    status = 'NO_NFT_REFERENCE'
                elif row_nft == next(iter(distinct_refs)):
                    status = 'MATCH'
                else:
                    status = 'MISMATCH'
                stats[(layer,status)] += 1
                if status in ('MISMATCH','CONFLICTING_NFT_FIELDS'):
                    target = next(iter(distinct_refs)) if len(distinct_refs)==1 else None
                    layers = sorted(asset_layers.get(target, [])) if target else []
                    entry = dict(
                        layer=layer,line_number=line_no,row_nft=row_nft,asset_name=r.get('asset_name',r.get('u')),
                        action_type=r.get('action_type',r.get('y')),action_id=r.get('action_id',r.get('aid')),
                        trace_id=r.get('trace_id',r.get('tid')),timestamp=r.get('timestamp',r.get('t')),
                        details_nft_item=refs.get('nft_item'),details_nft_address=refs.get('nft_address'),
                        status=status,target_in_dataset=bool(layers),target_layers=';'.join(layers),target_nft=target
                    )
                    writer.writerow(entry)
                    if len(examples[layer]) < 10:
                        examples[layer].append(entry)

summary = pd.DataFrame([{'layer':layer,'status':status,'n_actions':count}
                        for (layer,status),count in stats.items()])
display(summary.sort_values(['layer','status']))
print('Report salvato:', report_path.resolve())

,layer,status,n_actions
2,phone_numbers,MATCH,834748
3,phone_numbers,MISMATCH,2398
0,usernames,MATCH,2572141
1,usernames,MISMATCH,7012


Report salvato: /home/local/ADUNIPI/a.belliani/CNA/cleaning/history_audit/nft_mismatch_report.csv


## 3. Ispezione delle anomalie e conteggio per tipo

`NO_NFT_REFERENCE` non significa che la action sia estranea; per i tipi filtrati ci aspettiamo normalmente `nft_item`. Se compaiono mismatch, esaminarli *prima* di eliminare/spostare eventi. L'assenza di mismatch **non garantisce** la completezza storica della raccolta.

In [4]:
for layer in FILES:
    print('\n',layer.upper())
    display(pd.DataFrame(examples[layer]))

# Legge il report (non i dataset enormi)
report_df = pd.read_csv(report_path)
if not report_df.empty:
    display(report_df.groupby(['layer','status','action_type','target_layers'], dropna=False).size().reset_index(name='n').sort_values('n',ascending=False).head(30))
else:
    print('Nessun mismatch esplicito rilevato.')


 USERNAMES


,layer,line_number,row_nft,asset_name,action_type,action_id,trace_id,timestamp,details_nft_item,details_nft_address,status,target_in_dataset,target_layers,target_nft
0,usernames,66,0:e74830f3e72e2828303bf93f4bb4498e6e5c8b4c6321...,@plita,nft_transfer,Ujd6KBwAyu9A2xX8PIIiblPgsTuLOOrDqhdNJboZfpM=,iR2thXgPNA6P9WUm37SSjdQoR7vufD1zhh3fcisFP/U=,2025-01-17T08:45:39Z,0:07a01f587351bf3a048ad92126067c33658c42135a35...,None,MISMATCH,False,,0:07a01f587351bf3a048ad92126067c33658c42135a35...
1,usernames,145,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,j04nyT4/FQa3tMMzDTCyQQ95woJ461IYNXWo3Iyyd80=,Q1+3v2ALKabo3N1y2lzfKRmz7cJDHXv0EWaQrYzdmyE=,2025-01-09T04:45:22Z,0:f94d87709408a76e28898a75c91467134110b8b3a824...,None,MISMATCH,False,,0:f94d87709408a76e28898a75c91467134110b8b3a824...
2,usernames,146,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,ytiOYKKTnm/rxBbLbwzgNvbys9jBIFJkhBqgNthpWFc=,1YtWLYaAeWRvO8Zjve2fQyn9qdSHxf2JTPvDGfoqm9c=,2025-01-09T04:45:59Z,0:b93eca7b6dd7d8af3676cb51966e319279fdb54b3c28...,None,MISMATCH,False,,0:b93eca7b6dd7d8af3676cb51966e319279fdb54b3c28...
3,usernames,147,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,Cs9U2Rvzlt0i6MIyHJvO9uqKJgZUEiW7SvBmO6aJpR4=,ZsUu+ed9IMDDLk7WmAclgWgwnDNiYXxt6v0S1obbV3M=,2025-01-16T01:59:50Z,0:ef9ebfcd1fae1c852f026e3c895efb2452c76804789e...,None,MISMATCH,False,,0:ef9ebfcd1fae1c852f026e3c895efb2452c76804789e...
4,usernames,159,0:d66a6b198b665dd0d787eca8a251f7f2026ec8e9fa3e...,@ferrara,nft_transfer,lE0Jf04/ummuHs3Q51hK9g+R6ZoQHPlunT7nkNyfymw=,oHMtT1IEJ4SRMZ7MoDL1myHf8MchZuOVcvmbFnsvAxQ=,2024-09-24T12:04:56Z,0:a54bda345ffa79e03907985679ca68d2f4e149ec711d...,None,MISMATCH,False,,0:a54bda345ffa79e03907985679ca68d2f4e149ec711d...
5,usernames,281,0:e3770586c85c1b77fc61d4a3e4440bbd15c8640c9140...,@gt1111,nft_transfer,3/o4KfO9tgCiOYJYYbI5RVtAN4AWX27jLtuqef1cI5w=,NnaVPF9Z0EVP2EAjepL45SSMwfkIpD8zbQJVyGuVO/k=,2024-12-26T02:11:32Z,0:e5892b380d4ce9ff2c7de7782131d471c6957a63aff4...,None,MISMATCH,False,,0:e5892b380d4ce9ff2c7de7782131d471c6957a63aff4...
6,usernames,288,0:c2e117ae8c9ef2eb1d55b2eb14da31b914e3d1b4ad85...,@letschange,nft_transfer,XtKTfj7PUnlwXgp0DQ29m7iE83MO2UQNqV++K3zDCVc=,/5krtAxjI+ZPQqC9iUGIxGWQqpv3eTYhpFUzg0lrCzI=,2024-06-15T20:24:34Z,0:9a5f2d9005c4c42cde09f71c0b3e70de6f1ac2307d7b...,None,MISMATCH,False,,0:9a5f2d9005c4c42cde09f71c0b3e70de6f1ac2307d7b...
7,usernames,1277,0:720a2c784a8dd1cd951fb5648314cea1d9ece791588b...,@risse,nft_transfer,hBzL+2csXdNG9Zz+4jEKH7C2juncOuAnMD5BDwxfQuo=,I/6n1g4dcSqPoeXexE02IPYnN5i7gE5eX9qOt3njQBY=,2024-07-30T00:46:24Z,0:3808cba2e28f505d626f6ad1d4013b24480ed464fd90...,None,MISMATCH,False,,0:3808cba2e28f505d626f6ad1d4013b24480ed464fd90...
8,usernames,1912,0:2ef0962d174df7c0599bb9d13bb3598b47ce12e6b4e9...,@goat365,nft_transfer,7zC7Y2+eq+zYHJLFU5n7oLc1DpLyZBsh7v3/pZIy9G8=,ILrC+BlV/d5qq05ia9XGEWIrd+w9PnxLJzazAUySVSE=,2024-08-15T22:40:04Z,0:3c192c875ccabc3d4a67e2d1142269ee8c89149f326b...,None,MISMATCH,False,,0:3c192c875ccabc3d4a67e2d1142269ee8c89149f326b...
9,usernames,2269,0:4005779fc4c5749da6722c839dddb02bea6bc745ffe2...,@qwertyksc,nft_transfer,YDTK2+vVv1zGugIQp3JuITujiB1jTDJA8hQIkXZ3oU8=,HC7YMkNENP+qF3z+L9LEfqytdlPckBhVujn/PGW9m/o=,2024-09-16T10:30:01Z,0:bafa1c7b89e3e44a5a5f2c6915f18ae2071bb45db13a...,None,MISMATCH,False,,0:bafa1c7b89e3e44a5a5f2c6915f18ae2071bb45db13a...



 PHONE_NUMBERS


,layer,line_number,row_nft,asset_name,action_type,action_id,trace_id,timestamp,details_nft_item,details_nft_address,status,target_in_dataset,target_layers,target_nft
0,phone_numbers,17,0:4ec906c013aea2bf18b90c5323544a52de08c34e52d5...,+888 0407 3815,nft_transfer,6y7isoXAX8FcToig/0P775Zs/zEoE8MFd+KmXhNxxH8=,iA6I7cbQITDyt+jWNmBtm1vQsujwcwYuSYpOPKNTQVs=,2024-11-02T21:18:48Z,0:62658d0d75e3eb8922d9c2fed6c1b5ff34ac82cc9881...,None,MISMATCH,False,,0:62658d0d75e3eb8922d9c2fed6c1b5ff34ac82cc9881...
1,phone_numbers,25,0:4766fe62d643e7a49276c82d206531b46cdd81876132...,+888 0496 2397,nft_transfer,MzXa8+Ivhe4zO0Bdkx3upSEWvHZgz8/oGPGfE7fLAKM=,elVKoqns8ERSYXA1w8EkqCERRwGfZC9exKjZxuRqvRQ=,2024-11-02T21:18:36Z,0:fb0c4e84a6b85477fb78ee13052dc61dec34c5f17f9d...,None,MISMATCH,False,,0:fb0c4e84a6b85477fb78ee13052dc61dec34c5f17f9d...
2,phone_numbers,29,0:2908c490072e085db6663fd233245687eb120eca00dd...,+888 0946 2753,nft_transfer,yXqKf81RXUhlK/2emAgCK15QAxjmd1hZDse+U75B/6M=,Fig/vWxaPId84iX8bqHZjNPCaxeAVqVGBu4Tmjyxm00=,2024-11-02T21:17:10Z,0:edb6be391711f8b06f52386ea4d4ce00bf6b5667b5f8...,None,MISMATCH,False,,0:edb6be391711f8b06f52386ea4d4ce00bf6b5667b5f8...
3,phone_numbers,33,0:b8053b8153eaf1930c363a917aa14adcf8e95ccc952a...,+888 0135 4796,nft_transfer,le7AK2FPPG+8D6sxPy2N43MwmZIkmHhcWzE4b8M8XdU=,lnCO67UUaPEVkYN25xY5dJxoC+PgjHqHgT7yhnNRFys=,2024-09-24T10:58:40Z,0:8b761ca6792b901bc7f33f956a1efbcba703f3430c84...,None,MISMATCH,False,,0:8b761ca6792b901bc7f33f956a1efbcba703f3430c84...
4,phone_numbers,43,0:335e2a18af2e72405dd0f1892183a851515d2badc425...,+888 0839 7064,nft_transfer,BIbLxNZtJxQZXN+CMUBxfNV9CX7TTXtKUVedmNo+xxk=,sE4rI59a0NGoSnDyfXs8zXtPad8XCLsKulfemRQzBH8=,2024-11-02T21:17:48Z,0:a40697b2366003ba10a3c4a3f230f3240585fb2cbcd2...,None,MISMATCH,False,,0:a40697b2366003ba10a3c4a3f230f3240585fb2cbcd2...
5,phone_numbers,55,0:5534a0ceec87552a27fc4a1e478e6fc44640a61cf1d7...,+888 0437 1624,nft_transfer,ZnWRoKdtre5UXa3coeFY2kObe2CnivE91ObfyCoUIqQ=,tJOlKCdx7jgB4gZnnOzQ82gUzt55jk8donwzpaVRFb4=,2024-11-02T21:19:04Z,0:706609c171dedae1f99facebfc9eec41646dedb5eabe...,None,MISMATCH,False,,0:706609c171dedae1f99facebfc9eec41646dedb5eabe...
6,phone_numbers,57,0:57582ea4b33e2ffd2d513ab19c4e047b579a85164378...,+888 0013 0379,nft_transfer,dHiPyjghhIxemBzirpiP96atPwV8Gd4W1asu88gSoVM=,2uhPqOp9mIWIItK5lGGPmjRorvpEaZXQEvTweBsU4MQ=,2024-11-03T18:29:28Z,0:6279810405a220f1852790e2e16be11710366e81d5f9...,None,MISMATCH,False,,0:6279810405a220f1852790e2e16be11710366e81d5f9...
7,phone_numbers,121,0:007a6cf82ae2c2e2889956fb241af6910ff9c2583f32...,+888 0526 3517,nft_transfer,tLJPvWMoyoH4d6gafc2B51Xwzw+dr6EMpALPeXMJ52k=,YqA98bawZiSTSi8w3RANr7a+tb8L+dMIs3K7bhBG5Ks=,2024-11-02T03:29:22Z,0:cd93e4c9c224daf45fb641c0d113dc567d1048947df2...,None,MISMATCH,False,,0:cd93e4c9c224daf45fb641c0d113dc567d1048947df2...
8,phone_numbers,122,0:007a6cf82ae2c2e2889956fb241af6910ff9c2583f32...,+888 0526 3517,nft_transfer,/YvXpLfMAFb2zlGKusCwPB7y4SJKFwpuo0WmYL82QoU=,G+GsIU/Irf0uYLbQ6IE7B1yTOkb59yGz91ygzP5PJY0=,2024-11-02T17:16:48Z,0:a9d3ea951bf0ace1693769115ab1a9d25aeff381b7ee...,None,MISMATCH,False,,0:a9d3ea951bf0ace1693769115ab1a9d25aeff381b7ee...
9,phone_numbers,123,0:007a6cf82ae2c2e2889956fb241af6910ff9c2583f32...,+888 0526 3517,nft_transfer,urQVYW4zcE11fSoGz17eLYZOpqhBjeADrY3YfcuGOuU=,NbYlcLgZZuMx9wQg6CRjavxf4q3lWzrBVkywTCtaojM=,2024-11-02T17:38:23Z,0:7eeeaee9b83b8c9149a30c2e56e4178e209b349c18e2...,None,MISMATCH,False,,0:7eeeaee9b83b8c9149a30c2e56e4178e209b349c18e2...


,layer,status,action_type,target_layers,n
4,usernames,MISMATCH,nft_transfer,NaN,6943
2,phone_numbers,MISMATCH,nft_transfer,NaN,2364
3,usernames,MISMATCH,nft_transfer,usernames,69
0,phone_numbers,MISMATCH,nft_transfer,phone_numbers,30
1,phone_numbers,MISMATCH,nft_transfer,usernames,4


## 4. Esaminare il record completo di una riga sospetta (opzionale)

Questa funzione usa il numero della riga nel CSV di report e riapre soltanto la riga selezionata dal JSONL. La ricerca è sequenziale e non carica l'intero file.

In [5]:
from itertools import islice
from pprint import pprint

def mostra_record(layer, line_number):
    path = FILES[layer]
    with path.open('r', encoding='utf-8') as f:
        line = next(islice(f, int(line_number)-1, int(line_number)), None)
    if line is None:
        raise ValueError('Riga non trovata')
    pprint(json.loads(line), sort_dicts=False)

# Esempio (solo se ci sono mismatch):
# r = report_df.iloc[0]
# mostra_record(r['layer'], r['line_number'])

### Nota metodologica

Il notebook precedente lavorava su eventi TonAPI annidati (`history` → `actions` → `type` → `nft`) e utilizzava `base_transactions` per deduplicazione/spostamento. Qui i dati derivano dalle actions TON Center e sono già `nft_transfer`/`nft_purchase` filtrati. Il confronto `nft` ↔ `details.nft_item` è un **controllo di coerenza**, non può rilevare tutte le azioni estranee che potrebbero essere state perse o attribuite in fase di estrazione. Non usiamo `nft_collection`, perché identifica la collezione, non il singolo NFT.

In [6]:
import pandas as pd
from pathlib import Path

PATH = Path("cleaning/history_audit/nft_mismatch_report.csv")

df = pd.read_csv(PATH)

print("Colonne disponibili:")
print(df.columns.tolist())

print("\nNumero totale di anomalie:", len(df))

display(df.head(10))

Colonne disponibili:
['layer', 'line_number', 'row_nft', 'asset_name', 'action_type', 'action_id', 'trace_id', 'timestamp', 'details_nft_item', 'details_nft_address', 'status', 'target_in_dataset', 'target_layers', 'target_nft']

Numero totale di anomalie: 9410


,layer,line_number,row_nft,asset_name,action_type,action_id,trace_id,timestamp,details_nft_item,details_nft_address,status,target_in_dataset,target_layers,target_nft
0,usernames,66,0:e74830f3e72e2828303bf93f4bb4498e6e5c8b4c6321...,@plita,nft_transfer,Ujd6KBwAyu9A2xX8PIIiblPgsTuLOOrDqhdNJboZfpM=,iR2thXgPNA6P9WUm37SSjdQoR7vufD1zhh3fcisFP/U=,2025-01-17T08:45:39Z,0:07a01f587351bf3a048ad92126067c33658c42135a35...,NaN,MISMATCH,False,NaN,0:07a01f587351bf3a048ad92126067c33658c42135a35...
1,usernames,145,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,j04nyT4/FQa3tMMzDTCyQQ95woJ461IYNXWo3Iyyd80=,Q1+3v2ALKabo3N1y2lzfKRmz7cJDHXv0EWaQrYzdmyE=,2025-01-09T04:45:22Z,0:f94d87709408a76e28898a75c91467134110b8b3a824...,NaN,MISMATCH,False,NaN,0:f94d87709408a76e28898a75c91467134110b8b3a824...
2,usernames,146,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,ytiOYKKTnm/rxBbLbwzgNvbys9jBIFJkhBqgNthpWFc=,1YtWLYaAeWRvO8Zjve2fQyn9qdSHxf2JTPvDGfoqm9c=,2025-01-09T04:45:59Z,0:b93eca7b6dd7d8af3676cb51966e319279fdb54b3c28...,NaN,MISMATCH,False,NaN,0:b93eca7b6dd7d8af3676cb51966e319279fdb54b3c28...
3,usernames,147,0:ffb2e9cffae5342f3966c0d688b9c1b9324e74b29e41...,@dinex,nft_transfer,Cs9U2Rvzlt0i6MIyHJvO9uqKJgZUEiW7SvBmO6aJpR4=,ZsUu+ed9IMDDLk7WmAclgWgwnDNiYXxt6v0S1obbV3M=,2025-01-16T01:59:50Z,0:ef9ebfcd1fae1c852f026e3c895efb2452c76804789e...,NaN,MISMATCH,False,NaN,0:ef9ebfcd1fae1c852f026e3c895efb2452c76804789e...
4,usernames,159,0:d66a6b198b665dd0d787eca8a251f7f2026ec8e9fa3e...,@ferrara,nft_transfer,lE0Jf04/ummuHs3Q51hK9g+R6ZoQHPlunT7nkNyfymw=,oHMtT1IEJ4SRMZ7MoDL1myHf8MchZuOVcvmbFnsvAxQ=,2024-09-24T12:04:56Z,0:a54bda345ffa79e03907985679ca68d2f4e149ec711d...,NaN,MISMATCH,False,NaN,0:a54bda345ffa79e03907985679ca68d2f4e149ec711d...
5,usernames,281,0:e3770586c85c1b77fc61d4a3e4440bbd15c8640c9140...,@gt1111,nft_transfer,3/o4KfO9tgCiOYJYYbI5RVtAN4AWX27jLtuqef1cI5w=,NnaVPF9Z0EVP2EAjepL45SSMwfkIpD8zbQJVyGuVO/k=,2024-12-26T02:11:32Z,0:e5892b380d4ce9ff2c7de7782131d471c6957a63aff4...,NaN,MISMATCH,False,NaN,0:e5892b380d4ce9ff2c7de7782131d471c6957a63aff4...
6,usernames,288,0:c2e117ae8c9ef2eb1d55b2eb14da31b914e3d1b4ad85...,@letschange,nft_transfer,XtKTfj7PUnlwXgp0DQ29m7iE83MO2UQNqV++K3zDCVc=,/5krtAxjI+ZPQqC9iUGIxGWQqpv3eTYhpFUzg0lrCzI=,2024-06-15T20:24:34Z,0:9a5f2d9005c4c42cde09f71c0b3e70de6f1ac2307d7b...,NaN,MISMATCH,False,NaN,0:9a5f2d9005c4c42cde09f71c0b3e70de6f1ac2307d7b...
7,usernames,1277,0:720a2c784a8dd1cd951fb5648314cea1d9ece791588b...,@risse,nft_transfer,hBzL+2csXdNG9Zz+4jEKH7C2juncOuAnMD5BDwxfQuo=,I/6n1g4dcSqPoeXexE02IPYnN5i7gE5eX9qOt3njQBY=,2024-07-30T00:46:24Z,0:3808cba2e28f505d626f6ad1d4013b24480ed464fd90...,NaN,MISMATCH,False,NaN,0:3808cba2e28f505d626f6ad1d4013b24480ed464fd90...
8,usernames,1912,0:2ef0962d174df7c0599bb9d13bb3598b47ce12e6b4e9...,@goat365,nft_transfer,7zC7Y2+eq+zYHJLFU5n7oLc1DpLyZBsh7v3/pZIy9G8=,ILrC+BlV/d5qq05ia9XGEWIrd+w9PnxLJzazAUySVSE=,2024-08-15T22:40:04Z,0:3c192c875ccabc3d4a67e2d1142269ee8c89149f326b...,NaN,MISMATCH,False,NaN,0:3c192c875ccabc3d4a67e2d1142269ee8c89149f326b...
9,usernames,2269,0:4005779fc4c5749da6722c839dddb02bea6bc745ffe2...,@qwertyksc,nft_transfer,YDTK2+vVv1zGugIQp3JuITujiB1jTDJA8hQIkXZ3oU8=,HC7YMkNENP+qF3z+L9LEfqytdlPckBhVujn/PGW9m/o=,2024-09-16T10:30:01Z,0:bafa1c7b89e3e44a5a5f2c6915f18ae2071bb45db13a...,NaN,MISMATCH,False,NaN,0:bafa1c7b89e3e44a5a5f2c6915f18ae2071bb45db13a...


In [7]:
import pandas as pd

df = pd.read_csv(
    "cleaning/history_audit/nft_mismatch_report.csv"
)

print("Mismatch per action type:")
display(
    df.groupby(["layer", "action_type"])
      .size()
      .reset_index(name="n_actions")
)

print("\nTarget presenti o assenti nel dataset:")
display(
    df.groupby(["layer", "target_in_dataset"], dropna=False)
      .size()
      .reset_index(name="n_actions")
)

print("\nTarget in altri layer:")
display(
    df.groupby(["layer", "target_layers"], dropna=False)
      .size()
      .reset_index(name="n_actions")
)

Mismatch per action type:


,layer,action_type,n_actions
0,phone_numbers,nft_transfer,2398
1,usernames,nft_transfer,7012



Target presenti o assenti nel dataset:


,layer,target_in_dataset,n_actions
0,phone_numbers,False,2364
1,phone_numbers,True,34
2,usernames,False,6943
3,usernames,True,69



Target in altri layer:


,layer,target_layers,n_actions
0,phone_numbers,phone_numbers,30
1,phone_numbers,usernames,4
2,phone_numbers,NaN,2364
3,usernames,usernames,69
4,usernames,NaN,6943


In [15]:
import pandas as pd
from pathlib import Path

REPORT = Path("cleaning/history_audit/nft_mismatch_report.csv")

FILES = {
    "usernames": {
        "input": Path("dataset_normalized/usernames_normalized.jsonl"),
        "output": Path("dataset_clean_final/usernames_clean.jsonl")
    },
    "phone_numbers": {
        "input": Path("dataset_normalized/numbers_normalized.jsonl"),
        "output": Path("dataset_clean_final/numbers_clean.jsonl")
    }
}

Path("dataset_clean_final").mkdir(parents=True, exist_ok=True)

df_mismatch = pd.read_csv(REPORT)

display(df_mismatch.groupby("layer").size())

layer
phone_numbers    2398
usernames        7012
dtype: int64

In [16]:
def rimuovi_mismatch(layer, input_path, output_path):

    # Numeri di riga indicati dal report di audit
    righe_da_eliminare = set(
        df_mismatch.loc[
            (df_mismatch["layer"] == layer) &
            (df_mismatch["status"] == "MISMATCH"),
            "line_number"
        ].astype(int)
    )

    totale = 0
    mantenute = 0
    eliminate = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line_number, line in enumerate(fin, 1):
            totale += 1

            if line_number in righe_da_eliminare:
                eliminate += 1
                continue

            fout.write(line)
            mantenute += 1

    print(f"\n{layer.upper()}")
    print(f"Righe originali: {totale:,}")
    print(f"Righe eliminate: {eliminate:,}")
    print(f"Righe mantenute: {mantenute:,}")

    assert eliminate == len(righe_da_eliminare), (
        "Il numero di righe eliminate non coincide con il report!"
    )

In [17]:
for layer, paths in FILES.items():
    rimuovi_mismatch(
        layer,
        paths["input"],
        paths["output"]
    )


USERNAMES
Righe originali: 2,579,153
Righe eliminate: 7,012
Righe mantenute: 2,572,141

PHONE_NUMBERS
Righe originali: 837,146
Righe eliminate: 2,398
Righe mantenute: 834,748


In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

FILES = {
    "usernames": Path("dataset_clean_final/usernames_clean.jsonl"),
    "phone_numbers": Path("dataset_clean_final/numbers_clean.jsonl")
}